# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FARNHELL/ML-INTERNSHIP/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I will start with **Logistic Regression**. The model predicts a probability that a page will have an observed forward decline; sorting that probability gives the editor the ranked refresh queue they need when time is limited. It can use the five Week 3 leakage-free features — `feat_impr_7d`, `feat_clicks_7d`, `feat_pos_7d`, `feat_scroll_7d`, and `content_type` — without using product flags, traffic-source fields, or future-window data.

Logistic Regression is deliberately a simple first model: its coefficients make it possible to see which signals are moving a page up or down the queue. A more complex model is not automatically better here; it earns a place only if it improves the same held-out precision@K and ROC-AUC comparison against the Week 4 rule, not just because it is harder to explain.

The quick one-feature check printed after the frame loads shows that each numeric feature is weak by itself (correlations from −0.020 to −0.088, with `feat_scroll_7d` closest to zero). `content_type` has a clearer difference: comparison articles have a 0.844 decline rate versus 0.568 for feedly and keyword articles; this is plausible enough to test a simple linear combination, but not strong enough to skip held-out evaluation.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I use a grouped client split with `GroupShuffleSplit`: 75% of clients for training and 25% for test, with `random_state=42`. Pages from the same client share reporting and content patterns, so a random row split would let those client-level patterns appear on both sides and make the result look better than it is. This follows the Week 3 leakage care: `client_hash_id` is used for grouping only, never as a feature.

This fixed split places 24 clients and 21,887 rows in training, and 8 clients and 2,049 rows in test. The client target is 75/25, but the row counts are not close to 75/25 because a few clients contribute most of the eligible rows; that imbalance is reported rather than hidden.


In [1]:
from pathlib import Path
import os
import tempfile

import pandas as pd
from sklearn.model_selection import GroupShuffleSplit


def find_repo_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'data' / 'raw' / 'content_refresh_anonymized.csv').exists():
            return candidate
    raise FileNotFoundError('Run this notebook from inside ML-INTERNSHIP.')


ROOT = find_repo_root()
OUTPUT_DIR = ROOT / 'work' / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FEATURES = ['feat_impr_7d', 'feat_clicks_7d', 'feat_pos_7d', 'feat_scroll_7d', 'content_type']
NUMERIC_FEATURES = FEATURES[:4]
CATEGORICAL_FEATURES = ['content_type']


def load_hf_token():
    try:
        from google.colab import userdata
        return userdata.get('HF_TOKEN').strip()
    except (ImportError, KeyError, AttributeError):
        try:
            from dotenv import load_dotenv
            load_dotenv(ROOT / '.env')
        except ImportError:
            pass
        return os.environ.get('HF_TOKEN', '').strip()


def warehouse_file(filename, token):
    from huggingface_hub import hf_hub_download
    try:
        return hf_hub_download('FlyRank/internship-warehouse', filename=filename, repo_type='dataset', token=token)
    except PermissionError:
        return hf_hub_download('FlyRank/internship-warehouse', filename=filename, repo_type='dataset', token=token, local_dir=Path(tempfile.gettempdir()) / 'flyrank_w05_warehouse')


def load_warehouse_frame():
    token = load_hf_token()
    if not token:
        raise RuntimeError('HF_TOKEN is not configured')
    daily_columns = ['report_date', 'client_hash_id', 'content_hash_id', 'gsc_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'scroll_events']
    march = pd.read_parquet(warehouse_file('fact_content_daily_performance/month=2026-03/data_0.parquet', token), columns=daily_columns)
    april = pd.read_parquet(warehouse_file('fact_content_daily_performance/month=2026-04/data_0.parquet', token), columns=['client_hash_id', 'content_hash_id', 'gsc_data_available', 'gsc_impressions'])
    content = pd.read_parquet(warehouse_file('dim_content.parquet', token), columns=['client_hash_id', 'content_hash_id', 'content_updated_date', 'content_type'])
    march['report_date'] = pd.to_datetime(march['report_date'])
    march = march[march['gsc_data_available'].astype(bool)].copy()
    april = april[april['gsc_data_available'].astype(bool)].copy()
    keys = ['client_hash_id', 'content_hash_id']
    march_total = march.groupby(keys, as_index=False).agg(march_impressions_30d=('gsc_impressions', 'sum'))
    features = march[march['report_date'].ge('2026-03-25')].groupby(keys, as_index=False).agg(feat_impr_7d=('gsc_impressions', 'mean'), feat_clicks_7d=('gsc_clicks', 'mean'), feat_pos_7d=('gsc_avg_position', 'mean'), feat_scroll_7d=('scroll_events', 'mean'))
    april_total = april.groupby(keys, as_index=False).agg(april_impressions_30d=('gsc_impressions', 'sum'))
    content['content_updated_date'] = pd.to_datetime(content['content_updated_date'], errors='coerce')
    frame = march_total.merge(features, on=keys).merge(april_total, on=keys).merge(content, on=keys, how='left')
    frame['days_since_last_update'] = (pd.Timestamp('2026-03-31') - frame['content_updated_date']).dt.days
    # Match the Week 4 eligible warehouse cohort; the date itself is not a model feature.
    frame = frame[frame['days_since_last_update'].ge(0)].copy()
    frame['observed_forward_decline'] = (frame['march_impressions_30d'].gt(0) & frame['april_impressions_30d'].lt(.8 * frame['march_impressions_30d'])).astype(int)
    frame['data_source'] = 'warehouse: March features, April held-out label'
    return frame


try:
    model_frame = load_warehouse_frame()
    print('Using warehouse data for training and evaluation.')
except Exception as warehouse_error:
    starter = pd.read_csv(ROOT / 'data' / 'raw' / 'content_refresh_anonymized.csv')
    model_frame = starter.assign(feat_impr_7d=starter['impressions_90d'] / 90, feat_clicks_7d=starter['clicks_90d'] / 90, feat_pos_7d=starter['avg_position'], feat_scroll_7d=starter['scroll_events_90d'] / 90, observed_forward_decline=starter['trend_direction'].eq('down').astype(int), data_source='starter CSV fallback')
    print('Warehouse unavailable; starter fallback used: {}'.format(type(warehouse_error).__name__))

# Quick Section 1 signal check: each feature by itself versus the held-out outcome.
label = model_frame['observed_forward_decline']
signal_table = pd.DataFrame({
    'feature': NUMERIC_FEATURES,
    'point_biserial_corr': [model_frame[feature].corr(label) for feature in NUMERIC_FEATURES],
    'mean_no_decline': [model_frame.loc[label.eq(0), feature].mean() for feature in NUMERIC_FEATURES],
    'mean_decline': [model_frame.loc[label.eq(1), feature].mean() for feature in NUMERIC_FEATURES],
})
content_type_signal = model_frame.assign(content_type=model_frame['content_type'].fillna('missing')).groupby('content_type', observed=True).agg(
    n=('observed_forward_decline', 'size'),
    decline_rate=('observed_forward_decline', 'mean'),
).sort_values('decline_rate', ascending=False)
print('\nQuick Section 1 signal check — one feature at a time, before fitting a model')
print(signal_table.to_string(index=False, formatters={'point_biserial_corr': '{:+.3f}'.format, 'mean_no_decline': '{:.3f}'.format, 'mean_decline': '{:.3f}'.format}))
print('\ncontent_type decline-rate table')
print(content_type_signal.to_string(formatters={'decline_rate': '{:.3f}'.format}))

splitter = GroupShuffleSplit(n_splits=1, test_size=.25, random_state=42)
train_index, test_index = next(splitter.split(model_frame, groups=model_frame['client_hash_id']))
train_frame = model_frame.iloc[train_index].copy()
test_frame = model_frame.iloc[test_index].copy()
print('Eligible rows: {:,}; base rate: {:.3f}'.format(len(model_frame), model_frame['observed_forward_decline'].mean()))
print('Train: {:,} rows from {} clients. Test: {:,} rows from {} clients.'.format(len(train_frame), train_frame['client_hash_id'].nunique(), len(test_frame), test_frame['client_hash_id'].nunique()))
print('Test base rate: {:.3f}'.format(test_frame['observed_forward_decline'].mean()))

Using warehouse data for training and evaluation.

Quick Section 1 signal check — one feature at a time, before fitting a model
       feature point_biserial_corr mean_no_decline mean_decline
  feat_impr_7d              -0.054          56.119       40.302
feat_clicks_7d              -0.088           0.138        0.062
   feat_pos_7d              -0.070          17.297       14.882
feat_scroll_7d              -0.020           0.030        0.025

content_type decline-rate table
                        n decline_rate
content_type                          
comparison article    243        0.844
feedly article        961        0.568
keyword article     22732        0.568
Eligible rows: 23,936; base rate: 0.571
Train: 21,887 rows from 24 clients. Test: 2,049 rows from 8 clients.
Test base rate: 0.694


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

The model uses only the five Week 3 leakage-free features and predicts `observed_forward_decline`, the same March-to-April outcome used in Week 4. Its test score is a probability, sorted high to low before calculating precision@K.

For the fair comparison, I re-apply the frozen Week 4 rule to these same held-out clients only: percentile rank of `days_since_last_update` plus percentile rank of `feat_impr_7d`. Both rows in the main table therefore use the same 2,049 test rows, April label, and precision@K / ROC-AUC calculations. The older Week 4 full-queue results remain below as reference only; they are not used to decide which method wins this split.


In [2]:
import json
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocess = ColumnTransformer([
    ('numeric', Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), NUMERIC_FEATURES),
    ('content_type', Pipeline([('impute', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), CATEGORICAL_FEATURES),
])
model = Pipeline([
    ('preprocess', preprocess),
    ('logistic_regression', LogisticRegression(max_iter=1000, random_state=42)),
])
model.fit(train_frame[FEATURES], train_frame['observed_forward_decline'])
test_scores = model.predict_proba(test_frame[FEATURES])[:, 1]
ranked_test = test_frame.assign(model_score=test_scores).sort_values('model_score', ascending=False).reset_index(drop=True)
KS = [10, 20, 50, 100, 500]
model_precision = {k: float(ranked_test.head(k)['observed_forward_decline'].mean()) for k in KS}
model_auc = float(roc_auc_score(ranked_test['observed_forward_decline'], ranked_test['model_score']))

# Re-apply the frozen Week 4 rule to these exact held-out rows only.
# Week 4 rule: percentile rank(days_since_last_update) + percentile rank(feat_impr_7d).
baseline_test = test_frame.copy()
baseline_test['staleness_rank'] = baseline_test['days_since_last_update'].rank(pct=True, method='average')
baseline_test['volume_rank'] = baseline_test['feat_impr_7d'].rank(pct=True, method='average')
baseline_test['baseline_score'] = baseline_test['staleness_rank'] + baseline_test['volume_rank']
ranked_baseline_test = baseline_test.sort_values(['baseline_score', 'feat_impr_7d'], ascending=False).reset_index(drop=True)
baseline_precision = {k: float(ranked_baseline_test.head(k)['observed_forward_decline'].mean()) for k in KS}
baseline_auc = float(roc_auc_score(ranked_baseline_test['observed_forward_decline'], ranked_baseline_test['baseline_score']))

comparison = pd.DataFrame([
    {'method': 'Week 4 rule (same held-out clients)', **{'precision@{}'.format(k): baseline_precision[k] for k in KS}, 'ROC-AUC': baseline_auc},
    {'method': 'Logistic Regression (same held-out clients)', **{'precision@{}'.format(k): model_precision[k] for k in KS}, 'ROC-AUC': model_auc},
])
print('Same-split test base rate: {:.3f} (n={:,})'.format(ranked_test['observed_forward_decline'].mean(), len(ranked_test)))
print(comparison.to_string(index=False, formatters={column: '{:.3f}'.format for column in comparison.columns if column != 'method'}))
model_wins = [str(k) for k in KS if model_precision[k] > baseline_precision[k]]
baseline_wins = [str(k) for k in KS if model_precision[k] < baseline_precision[k]]
auc_reading = 'higher' if model_auc > baseline_auc else 'lower'
print('Plain reading: Logistic Regression is higher at precision@{} and lower at precision@{}; its ROC-AUC is {} than the same-split rule. It does not clearly win everywhere.'.format(', '.join(model_wins) or 'none', ', '.join(baseline_wins) or 'none', auc_reading))

# Preserved only as historical context, not as a fair comparison row.
week4_full_queue_reference = {'base_rate': .571, 'precision_at_k': {'10': .700, '20': .600, '50': .480, '100': .390, '500': .338}, 'roc_auc': .506}
print('Week 4 full queue (reference only): {}'.format(week4_full_queue_reference))

model_metrics = {
    'data_source': str(model_frame['data_source'].iloc[0]),
    'split': {'random_state': 42, 'group_column': 'client_hash_id', 'train_rows': int(len(train_frame)), 'test_rows': int(len(test_frame)), 'train_clients': int(train_frame['client_hash_id'].nunique()), 'test_clients': int(test_frame['client_hash_id'].nunique())},
    'features': FEATURES,
    'label': 'observed_forward_decline: April impressions are >20% below March',
    'test_base_rate': float(ranked_test['observed_forward_decline'].mean()),
    'same_split_baseline': {'rule': 'staleness_rank + volume_rank', 'precision_at_k': {str(k): value for k, value in baseline_precision.items()}, 'roc_auc': baseline_auc},
    'logistic_regression': {'precision_at_k': {str(k): value for k, value in model_precision.items()}, 'roc_auc': model_auc},
    'week4_full_queue_reference_only': week4_full_queue_reference,
}
metrics_path = OUTPUT_DIR / 'w05_model_metrics.json'
metrics_path.write_text(json.dumps(model_metrics, indent=2), encoding='utf-8')
print('Wrote model metrics to {}'.format(metrics_path))

Same-split test base rate: 0.694 (n=2,049)
                                     method precision@10 precision@20 precision@50 precision@100 precision@500 ROC-AUC
        Week 4 rule (same held-out clients)        0.700        0.400        0.600         0.670         0.770   0.590
Logistic Regression (same held-out clients)        0.600        0.700        0.680         0.660         0.654   0.463
Plain reading: Logistic Regression is higher at precision@20, 50 and lower at precision@10, 100, 500; its ROC-AUC is lower than the same-split rule. It does not clearly win everywhere.
Week 4 full queue (reference only): {'base_rate': 0.571, 'precision_at_k': {'10': 0.7, '20': 0.6, '50': 0.48, '100': 0.39, '500': 0.338}, 'roc_auc': 0.506}
Wrote model metrics to C:\INTERNSHIP\ML-INTERNSHIP\work\outputs\w05_model_metrics.json


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The error review looks at false positives near the top of the model queue and false negatives near the bottom. These are the expensive cases for the editor: a false positive can spend a refresh hour on a page that did not decline, while a false negative can leave a decline unreviewed. The examples show the observed feature values rather than pretending the probability is an explanation by itself.

For Logistic Regression, standardized numeric coefficients and one-hot content-type coefficients show what the model leans on. They are associations in this training split, not causes. Unlike the Week 4 rule, this model does not use staleness at all, so it does not repeat the known OPPOSITE-staleness signal; the Week 4 volume check was MIXED, so any traffic feature that looks important here still needs to earn trust from held-out results.


In [3]:
example_columns = ['content_hash_id', 'client_hash_id', *FEATURES, 'model_score', 'observed_forward_decline']
false_positives = ranked_test[ranked_test['observed_forward_decline'].eq(0)].head(3)
false_negatives = ranked_test[ranked_test['observed_forward_decline'].eq(1)].tail(3).sort_values('model_score')
print('False positives: high-priority model picks that did not decline (n=3 shown)')
print(false_positives[example_columns].to_string(index=False))
print('\nFalse negatives: low-priority model picks that did decline (n=3 shown)')
print(false_negatives[example_columns].to_string(index=False))

feature_names = model.named_steps['preprocess'].get_feature_names_out()
coefficients = model.named_steps['logistic_regression'].coef_[0]
importance = pd.DataFrame({'feature': feature_names, 'coefficient': coefficients})
importance['absolute_coefficient'] = importance['coefficient'].abs()
importance = importance.sort_values('absolute_coefficient', ascending=False)
print('\nLargest standardized Logistic Regression coefficients')
print(importance.head(8).to_string(index=False, formatters={'coefficient': '{:+.3f}'.format, 'absolute_coefficient': '{:.3f}'.format}))
positive = importance.sort_values('coefficient', ascending=False).iloc[0]
negative = importance.sort_values('coefficient').iloc[0]
print('\nInterpretation: {} is the strongest upward association with the decline label in this training split; {} is the strongest downward association.'.format(positive['feature'], negative['feature']))
print('These coefficients are not causal. The held-out ROC-AUC and precision@K table above remain the test of whether those associations help an editor.')

False positives: high-priority model picks that did not decline (n=3 shown)
         content_hash_id          client_hash_id  feat_impr_7d  feat_clicks_7d  feat_pos_7d  feat_scroll_7d    content_type  model_score  observed_forward_decline
content_5fb3505159eb03fd client_cd12bcfd98942aa1          1.75             0.0     5.791667        0.750000 keyword article     0.753374                         0
content_4bc0672ea82ded6c client_cd12bcfd98942aa1          3.40             0.0     7.585714        0.400000 keyword article     0.679878                         0
content_e36cd9f1cd7acb86 client_3197e6291363b4db          2.00             0.0     4.666667        0.333333 keyword article     0.673116                         0

False negatives: low-priority model picks that did decline (n=3 shown)
         content_hash_id          client_hash_id  feat_impr_7d  feat_clicks_7d  feat_pos_7d  feat_scroll_7d    content_type  model_score  observed_forward_decline
content_b5bef91d1b43e20c client_f623b

## Self-check

Before I submit, I confirm each line honestly:

- [x] Every section above is filled — with code where data or evaluation is required, or with a clearly reasoned written justification.
- [x] The notebook runs top to bottom with no errors.
- [x] No client names, URLs, or private queries appear in the notebook or outputs.
- [x] My claims use careful words: observed, measured, directional, decision-support.
- [x] locally solved and commited through git instead of colab.